# Chapter 7: Agentic RAG with Self-Correction

**Estimated time: 15 minutes**

You build a RAG system. A user asks about your cancellation policy. The retriever pulls three documents. One of them is an old blog post from two years ago that still advertises the previous refund terms. The LLM reads it and returns an answer that is confidently wrong. The system has no way to know it failed. It retrieved once, generated once, and shipped the answer. In this notebook you fix that. You add a grader node that checks the answer, and a loop that retries when the check fails.

## What you will see in three minutes

The SkillAgents corpus contains two documents that disagree about refunds. `refund_policy.pdf` is the current authoritative policy, updated March 2026. It says Pro Annual refunds are pro-rated with a $50 processing fee and Team plan refunds are only available before the first cohort kickoff. `outdated_blog_post.md` is a marketing announcement from January 2024 that promises a 30-day full refund guarantee with no questions asked and no processing fee. The two sources contradict each other directly.

You are going to ask this deliberately vague question:

> Tell me about the full refund guarantee

A naive RAG pipeline will retrieve whatever is closest in embedding space. The phrase "full refund guarantee" is a near-perfect match for the marketing blog post, so the naive pipeline pulls the outdated blog, feeds it to the LLM, and returns the old marketing claim about a 30-day no-fee refund. That answer is wrong. The real policy says pro-rated and Team plan has its own cutoff.

Self-RAG fixes this by adding a grader node to the loop. After the first answer is generated, the grader checks whether the answer is grounded in the authoritative source for the question. If not, the pipeline rewrites the query and retries, with a retrieval bias toward the authoritative source. After one retry the self-RAG graph converges on the correct policy language from `refund_policy.pdf`.

You will see the naive baseline return the stale answer, then watch the self-RAG loop trace node by node as it catches the stale answer, rewrites, re-retrieves, and lands on the grounded answer.

## Install the dependencies

Run the next cell once. In Colab it installs the Python packages fresh, including `langgraph` which is new in this chapter. Locally it assumes you already ran `pip install -r requirements.txt` in your virtual environment.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Cleaning up any prior chromadb install (silent)...")
    !pip uninstall -y -q chromadb chroma-hnswlib langchain-chroma 2>/dev/null

    print("Installing dependencies (takes about 60 seconds)...")
    !pip install -q \
        langchain==0.3.14 \
        langchain-openai==0.2.14 \
        langchain-community==0.3.14 \
        langchain-text-splitters==0.3.5 \
        langgraph==0.2.61 \
        faiss-cpu==1.13.2 \
        pypdf==5.1.0 \
        langsmith==0.2.6 \
        pandas==2.2.2 \
        matplotlib==3.9.4
    print("Done.")
else:
    print("Local environment detected. Skipping pip install.")
    print("Make sure you have run 'pip install -r requirements.txt' in your venv.")

In [ ]:
import os, sys

if IN_COLAB:
    # Always force a fresh clone so updates on GitHub are picked up. Without
    # this, a stale repo from a prior session would keep running the old utils.
    !rm -rf rag-for-pms
    !git clone -q https://github.com/DDRXV/rag-for-pms.git
    os.chdir("rag-for-pms")
else:
    # Local Jupyter: already inside the repo. Walk up to root if we are in chapters/.
    if os.path.basename(os.getcwd()) == "chapters":
        os.chdir("..")

# Python caches imported modules in sys.modules. Drop any cached utils.* so the
# next import reads the freshly cloned file, not a stale copy from an earlier
# session.
for cached in [m for m in sys.modules if m.startswith("utils")]:
    del sys.modules[cached]

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
from utils.shared import get_keys
get_keys()

## What you just did

`get_keys` pulled your OpenAI, LangSmith, and Cohere keys out of Colab secrets and turned on LangSmith tracing for this notebook. Every LLM call and every retrieval call in this chapter gets logged automatically to a visual waterfall at [smith.langchain.com](https://smith.langchain.com). That is especially useful in this chapter because the self-RAG graph makes multiple LLM calls per question, and a trace view is much easier to follow than a wall of print output.

## Look at the corpus

The corpus is the same set of SkillAgents documents from earlier chapters. For this chapter the two that matter most are `refund_policy.pdf` (the current authoritative policy, updated March 2026) and `outdated_blog_post.md` (the January 2024 marketing post that announced a 30-day full refund guarantee). These two disagree with each other. The policy PDF is the source of truth.

In [ ]:
from utils.shared import load_corpus

docs = load_corpus(exclude_prefixes=['ch1_'])
for d in docs:
    print(f"  {d.metadata.get('source', 'unknown'):25s}  {len(d.page_content):5d} chars")

## Peek at the conflict

Read the two passages side by side before you run any retrieval. The blog post is marketing copy from January 2024. It promises a 30-day full refund with no processing fee. The refund policy PDF is the current document, updated March 2026. It says the Student plan is non-refundable once the first module is viewed, and Pro Annual refunds are pro-rated minus two months of service and a $50 processing fee.

The chatbot does not know which one is authoritative. It only sees two blobs of text in the same FAISS index. The embedding model is happy to hand over whichever one looks closer to the question.

In [ ]:
blog = next(d for d in docs if d.metadata["source"] == "outdated_blog_post.md")
policy = next(d for d in docs if d.metadata["source"] == "refund_policy.pdf")

print("--- outdated_blog_post.md (marketing, January 2024) ---")
print(blog.page_content[:500])
print()
print("--- refund_policy.pdf (current policy, updated March 2026) ---")
print(policy.page_content[:600])

## Build the index and pose the question

Reuse the chunking strategy from Chapter 1: `chunk_size=500` with `chunk_overlap=50`. That is the setting every earlier chapter has used. You are not changing retrieval this time. You are changing what happens after retrieval.

In [ ]:
from utils.shared import build_index

index = build_index(chunk_size=500, chunk_overlap=50, exclude_prefixes=['ch1_'])

question = "Tell me about the full refund guarantee"
print(f"Question: {question}")

## Naive RAG: retrieve, generate, ship

Before you build the self-correcting graph, watch the basic pipeline fail. Search the index with the user's question, feed the top three chunks into the LLM, print the answer. This is the same pattern every chapter from 1 to 6 has used.

In [ ]:
from utils.shared import search, show_results, generate_answer

naive_results = search(index, question, k=3)
show_results(naive_results, question=question)

## What the basic retriever grabbed

Look at the source column. All three top chunks come from `outdated_blog_post.md`. That is not a bug in the retriever. The blog post literally contains the phrases "30 days" and "full refund", so its embedding is the closest match to the user's question. The authoritative `refund_policy.pdf` talks about pro-rated refunds and processing fees, which do not embed near "full refund within 30 days".

Now feed those three stale chunks into the LLM and look at what the answer becomes.

In [ ]:
naive_answer = generate_answer(naive_results, question)
print("NAIVE ANSWER:")
print(naive_answer)

## Confidently wrong

Read the answer. The pipeline told the user they get a full refund within 30 days. That contradicts the current refund policy, which says the Student plan is non-refundable once the first module is viewed, and Pro Annual refunds are pro-rated minus two months of service with a $50 processing fee. A customer who trusts this answer and then tries to get their money back on day 29 is going to be upset, and support is going to get an angry ticket.

The basic RAG pipeline has no mechanism to catch this. It retrieved, it generated, it returned. No one checked whether the answer lines up with the authoritative source. That is the failure mode Self-RAG is designed to prevent.

## The Self-RAG idea

Self-RAG adds one new node after generation: a grader. The grader is an LLM prompted to score whether the answer is grounded in the retrieved context and consistent with the authoritative source. If the score is high, return the answer. If the score is low, rewrite the query, retrieve again, generate again, grade again. The loop is bounded by a maximum retry count so it always terminates.

Four nodes, four edges, one conditional branch. This is what the graph looks like:

```
             start
              |
              v
         +---------+
         | retrieve|<---+
         +---------+    |
              |         |
              v         |
         +---------+    |
         | generate|    |
         +---------+    |
              |         |
              v         |
         +---------+    |
         |  grade  |    |
         +---------+    |
          /       \    |
       pass       fail  |
        |          |    |
        v          v    |
       end     +---------+
               | rewrite |
               +---------+
```

The grader is the whole point. Without it the system has no feedback signal. With it the system can reject its own bad answers and try again.

## Grade the naive answer by hand

Before building the full graph, run the grader once on the naive answer you just got. This shows you exactly what the grader sees and exactly what verdict it returns. The grader is told that `refund_policy.pdf` is the authoritative source for refund questions. Any answer that conflicts with that source must be scored 1 or 2, even if the retrieved context seems to support the answer.

In [ ]:
from utils.shared import grade_answer

naive_context_docs = [doc for doc, _ in naive_results]
verdict = grade_answer(
    question,
    naive_answer,
    naive_context_docs,
    threshold=4,
)

print(f"Score: {verdict['score']} out of 5")
print(f"Passed: {verdict['passed']}")
print(f"Reason: {verdict['reason']}")

## What the grader saw

The grader returned a low score and a failure verdict. The reason mentions that the candidate answer conflicts with the authoritative refund policy, or that the authoritative source was not even present in the retrieved passages (because every chunk came from the outdated blog post). Either way, this is the signal you need. The system knows the answer is not trustworthy.

A naive RAG pipeline has no place for this signal. Self-RAG wires the signal into a conditional edge in the graph. A failure routes to a rewrite node, which then loops back to retrieve. A pass routes to the end. You are about to build that graph.

## Build the Self-RAG graph

`build_self_rag_graph` compiles a LangGraph state machine with the four nodes from the diagram above. The graph takes a FAISS index and returns a compiled object you can stream. Three parameters matter for the lesson:

- `max_retries`: how many times the loop is allowed to go back and try again. A value of 2 means the pipeline attempts retrieve, generate, grade up to three times total (the original attempt plus two retries) before it gives up and returns whatever it has.
- `grader_threshold`: the minimum score (1 to 5) that counts as a pass. A higher threshold means the grader is stricter and the loop retries more often.
- `k`: the number of chunks retrieved on each attempt. Same meaning as in every earlier chapter.

In [ ]:
from utils.shared import build_self_rag_graph

graph = build_self_rag_graph(
    index,
    max_retries=2,        # Try up to two retries after the first attempt
    grader_threshold=4,   # Require a grade of 4 or higher to pass
    k=3,                  # Retrieve 3 chunks per attempt
)
print("Self-RAG graph compiled with four nodes: retrieve, generate, grade, rewrite.")

## Run the loop

`run_self_rag` streams the graph execution and prints one block per node per attempt. Watch the trace below. Attempt 1 should retrieve from the outdated blog post, generate a stale answer, and fail the grader. The loop rewrites the query to name the official refund policy document, retrieves again, generates a new answer, and the grader should pass it.

In [ ]:
from utils.shared import run_self_rag

final_state = run_self_rag(graph, question)

## Read the trace

Every block in that trace is one node execution. The first attempt shows `outdated_blog_post.md` as every top source. The generate node produced the stale answer. The grader flagged it as a conflict with the authoritative source and returned a low score.

The rewrite node kicked in. It took the original question and produced a new version that explicitly references the official refund policy document. That rewritten query is what goes into the second retrieve attempt.

Attempt 2 hits different chunks. Sources should now include `refund_policy.pdf`. The generate node produces a new answer that matches the current policy. The grader passes it with a higher score and the loop terminates. The final answer is grounded in the authoritative document.

Compare the final answer to the naive answer you printed earlier. The naive answer said yes, full refund. The self-RAG answer reflects the real policy: pro-rated refunds, processing fees, plan-specific rules. One retry was the difference between confidently wrong and correctly grounded.

## Change the grader threshold and watch the loop

The grader threshold controls how strict the system is. A lower threshold means the grader accepts more answers and the loop retries less often. A higher threshold means the opposite. Try setting the threshold to 5 (the strictest possible) and see if the same question still converges within two retries. Then try 3 (more lenient) and see whether the naive stale answer sneaks through without a retry.

In [ ]:
# Stricter grader: nothing below a perfect score passes.
strict_graph = build_self_rag_graph(
    index,
    max_retries=2,
    grader_threshold=5,
    k=3,
)
strict_final = run_self_rag(strict_graph, question)

In [ ]:
# Lenient grader: anything at or above 3 passes. The stale answer might slip through.
lenient_graph = build_self_rag_graph(
    index,
    max_retries=2,
    grader_threshold=3,
    k=3,
)
lenient_final = run_self_rag(lenient_graph, question)

## Why the threshold matters

Setting the threshold is not about finding the one magic number. It is about the trade-off between false positives and false negatives.

- A strict threshold (5) catches more bad answers, at the cost of extra retries on answers that were already good enough. Each retry adds one retrieval call and two LLM calls, which costs latency and money.
- A lenient threshold (3) ships answers faster but lets borderline-wrong answers through. That is fine for low-stakes questions like "what time is the session" and dangerous for high-stakes ones like "how much do I get refunded".

In production you pick the threshold based on the cost of a wrong answer in your specific domain. A legal assistant gets a strict threshold. A product FAQ bot gets a lenient one. The numbers are a policy decision, not a technical one.

## Change the retry budget

The second knob is `max_retries`. That is the ceiling on how many times the loop can go back and try again after the first attempt. Setting it to 0 disables self-correction entirely, which is equivalent to the naive pipeline. Setting it to 1 gives the system one shot at self-correction. Setting it higher than 2 is usually a waste, because if a good answer did not show up in two tries it usually does not show up in four.

Try running with `max_retries=0` to confirm that the loop degrades back to the naive baseline, stale answer and all.

In [ ]:
# No retries at all. Whatever the first attempt produces is final.
no_retry_graph = build_self_rag_graph(
    index,
    max_retries=0,
    grader_threshold=4,
    k=3,
)
no_retry_final = run_self_rag(no_retry_graph, question)

## What the zero-retry run showed

The zero-retry graph retrieved the stale blog chunks, generated the stale answer, and graded it as a failure, but had no budget to do anything about the failure. The final answer is identical to the naive baseline from earlier in the notebook. This is the important point: the grader on its own does not fix anything. It is the retry loop that converts the grader's signal into a different answer.

A self-RAG pipeline without a retry budget is just a RAG pipeline with extra logging. The value shows up at the first retry, not at the first grade.

## Self-RAG vs CRAG

Self-RAG grades the answer after generation. Corrective RAG, usually called CRAG, grades the documents before generation. Same tool, different position in the pipeline.

Here is the difference. In CRAG, each retrieved chunk is scored for relevance before it reaches the LLM. High-confidence chunks go through. Low-confidence chunks get dropped. If every chunk scores low, CRAG falls back to a secondary source, usually web search. CRAG is a filter on the input. Self-RAG is a check on the output.

Think of CRAG as a bouncer at the door and Self-RAG as a reviewer after the meeting. CRAG prevents bad input. Self-RAG catches bad output. Production systems that care about correctness usually end up with both, because the failure modes are different. CRAG cannot catch a hallucination that was invented by the LLM out of thin air. Self-RAG cannot recover from retrieved chunks that are all irrelevant but plausibly worded. You need the pair.

## Retrieval loops are the general pattern

Self-RAG and CRAG are two named instances of a more general idea: retrieval loops. The system can run retrieval more than once in a single query. It can grade the retrieved documents, rewrite the query if they look weak, retrieve from a different data source, grade the final answer, and keep looping until something meets the quality bar or the retry budget runs out.

Most real queries still resolve on the first attempt. The loop is insurance for the hard cases: ambiguous questions, outdated documents, queries that span multiple topics. In this notebook the running question resolves in one retry. In practice you should expect about 70 to 85 percent of queries to resolve in one attempt, another 10 to 20 percent in two, and the remainder to either resolve in three or give up and return a "I do not have a confident answer" response to the user.

## When agentic RAG is worth the cost

Agentic RAG is not free. Every retry adds one retrieval call and two LLM calls (one to generate, one to grade). A basic RAG request takes one to two seconds. A self-RAG request with one retry takes four to six seconds. Two retries puts you near ten seconds. That is the difference between a chatbot that feels instant and one that feels slow.

The rule of thumb is simple: if a wrong answer costs more than a slow answer, use agentic RAG. If speed matters more than perfect accuracy, stick with basic RAG and invest in better chunking, better embeddings, or better retrieval techniques from earlier chapters.

Domains where the cost of a wrong answer is high include medical knowledge bases, legal document search, financial compliance, refund and billing policy questions (the example in this notebook), internal engineering runbooks, and any customer support flow where the cost of a rollback is high. Domains where speed usually wins include casual chat, search suggestions, and conversational brainstorming tools. Pick the knob that matches your product, not the one that looks impressive in a demo.

## Open your LangSmith trace

Open [smith.langchain.com](https://smith.langchain.com), click into the project called `rag-for-pms`, and open the most recent trace. You will see one waterfall per graph run, with each node as its own span: retrieve, generate, grade, rewrite, retrieve again, generate again, grade again. This is where the visual story lands for a PM. A trace view of a self-RAG loop is the single clearest way to explain what agentic RAG is doing to someone who has never seen one.

## What you can do at work on Monday

Self-correction is the line between a demo and a production system. Here are three questions to ask on your next RAG review.

1. What happens today if your retriever surfaces an outdated document? Does the pipeline ship the stale answer, or does something catch it? If the answer is "nothing catches it", your pipeline is the naive baseline from the top of this notebook.
2. Do you have a named authoritative source for the high-stakes topics in your corpus? Refund policy, security posture, data retention, compliance terms. If your team cannot point at one file per topic and say "this is the source of truth", the grader has nothing to compare against.
3. What is the latency budget for a single query? If the budget is under two seconds you probably cannot afford agentic RAG everywhere. Pick the top three highest-cost-of-wrong-answer query types and turn on self-correction for those only. Leave the rest on basic RAG.

A team that can answer these three questions has already made most of the decisions that matter. Turning on the LangGraph loop is the easy part.

## Try it yourself

Pick any of these. Change the value in the relevant cell, rerun, watch what happens.

1. Change the `question` variable from "Tell me about the full refund guarantee" to "Do you offer a no-questions-asked money back guarantee?" and rerun the naive pipeline and the self-RAG graph. Does the loop still catch the stale answer? Does it take one retry or two?
2. Change `grader_threshold=4` to `grader_threshold=5` in the `build_self_rag_graph` call and rerun. Does the graph now take two retries on some questions instead of one? Read the grader reasons to see what it is being strict about.
3. Change `max_retries=2` to `max_retries=3` and change the question to something your corpus has no authoritative answer for, like "What is your privacy policy?". Watch the loop use all three retries and still fail the grader. The final answer will include the reason field so the caller knows the answer is not trusted.

## Homework

Two exercises you can do on your own. Each takes about fifteen minutes.

1. **Apply this to your own company.** Find one topic in your product's help center where an old document still exists alongside a newer authoritative one. A deprecated pricing page, an old privacy policy, a blog post announcing a feature that has since changed. Write a user question that would naturally retrieve the old document, then decide what your authoritative source should be. You do not need to build the graph. You need the list of (question, wrong document, authoritative document) triples. That list is what your engineering team needs to pick grader thresholds and authoritative sources in production.

2. **Build the CRAG variant.** The current graph grades the answer after generation. Copy `build_self_rag_graph` in `utils/shared.py`, rename it to `build_crag_graph`, and add a new `grade_relevance` node between `retrieve` and `generate`. The node should ask an LLM to score each retrieved chunk on a 1 to 5 scale for relevance to the question, drop any chunk that scores below 3, and only pass the surviving chunks to the generate node. If every chunk gets dropped, route to a "no context" path that returns "I do not have a confident answer" instead of generating anything. Test it on the same refund question and see whether CRAG catches the stale answer on the first attempt, before generation, instead of catching it after the fact like Self-RAG does.

## What is next

This is the final chapter of the RAG for Product Managers series. Across seven chapters you built the basic retrieval pipeline (Chapter 1), rewrote vague queries into specific sub-questions (Chapter 2), routed questions to the right data source (Chapter 3), structured queries for structured data (Chapter 4), indexed documents in multiple representations (Chapter 5), reranked retrieved chunks for better precision (Chapter 6), and finally wrapped the whole thing in a self-correcting loop (this chapter). That is a complete picture of what a production RAG system actually looks like.

The path from here is not another chapter. It is the running system you have in front of you. Pick one RAG project at your own company, open a LangSmith trace on a real user question, and look at what your pipeline is actually doing at every step: chunking, retrieval, routing, reranking, generation, grading. You now have the vocabulary for every one of those stages. The next cohort of the SkillAgents AI course goes deeper on the production side of this (evaluation harnesses, dataset curation, cost monitoring, A/B testing of retrieval changes). If you want the live version of that material, the cohort starts every few weeks at skillagents.ai. If you want to build on your own, the bonus notebook on Pinecone (hosted at `chapters/bonus_pinecone.ipynb` when it lands) shows how to swap the in-memory FAISS index for a hosted vector database so this same pipeline can scale past one laptop.

Good luck out there. Go build something a product manager at your company actually uses.